# scRareBench | High-Level scDisInFact | Dataset 3 — Wu breast-cancer atlas

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

High-level flow:

`load_dataset(3) → MethodSpec(scDisInFact) → benchmark_method() → HTML / multi-seed ZIP`

Pinned revisions:
- scRareBench: `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`
- scDisInFact: `9466ecf257d923f879cb39712c530d24826d88bc`

Dataset 3 contract:
- 100,064 cells × 28,468 features
- label: `celltype_subset` (evaluation only)
- batch: `donor_id`
- canonical counts: `layers["counts"]`
- HVG policy: **4,000 global Seurat-v3 HVGs, span=0.3**
- scIB HVG policy: global

scDisInFact is a multi-batch/multi-condition method. The Dataset 3 benchmark
contract does not define an independent method condition, and using
`celltype_subset` would leak ground truth. This notebook therefore creates one
constant method-only condition (`baseline`) and evaluates only the batch-corrected
shared factor `mu_c`.

Use **GPU + High-RAM Colab**. scDisInFact densifies its input internally.
For a smoke test, temporarily change `METHOD_SEEDS` to `[42]`.

## 1. Install pinned scRareBench and scDisInFact

In [ ]:
import subprocess, sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
SCDISINFACT_COMMIT = "9466ecf257d923f879cb39712c530d24826d88bc"

SCRAREBENCH_GITHUB = (
    "git+https://github.com/amirhossein-alishahi/"
    f"scRareBench_.git@{SCRAREBENCH_REVISION}"
)
SCDISINFACT_GITHUB = (
    "git+https://github.com/ZhangLabGT/"
    f"scDisInFact.git@{SCDISINFACT_COMMIT}"
)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    SCRAREBENCH_GITHUB,
])

METHOD_RUNTIME_DEPENDENCIES = (
    "torch>=1.11.0",
    "scikit-learn>=0.22.1",
    "matplotlib>=3.5.2",
    "scikit-misc>=0.3",
)

from scrarebench.runtime import setup_runtime, print_install_report
report = setup_runtime(
    extra_requirements=METHOD_RUNTIME_DEPENDENCIES,
    extra_imports=("torch", "sklearn", "matplotlib", "skmisc"),
    quiet=False,
)
print_install_report(report)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    SCDISINFACT_GITHUB,
])

subprocess.check_call([
    sys.executable, "-c",
    (
        "import scDisInFact, torch; "
        "from scDisInFact import create_scdisinfact_dataset, scdisinfact; "
        "print('scDisInFact', getattr(scDisInFact, '__version__', 'unknown')); "
        "print('CUDA', torch.cuda.is_available())"
    ),
])


## 2. Load Dataset 3

In [ ]:
import torch
import scrarebench
from scrarebench import load_dataset, dataset_info

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU required. In Colab select Runtime → Change runtime type → GPU."
    )

adata = load_dataset(3)
info = dataset_info(adata)

expected = {
    "dataset_key": "wu_breast_cancer_atlas",
    "label_key": "celltype_subset",
    "batch_key": "donor_id",
    "count_layer": "counts",
    "scib_hvg_batch_mode": "global",
    "benchmark_ready": True,
}
for key, value in expected.items():
    if info.get(key) != value:
        raise RuntimeError(
            f"Dataset 3 contract mismatch for {key!r}: "
            f"{info.get(key)!r} != {value!r}"
        )

if adata.shape != (100_064, 28_468):
    raise RuntimeError(f"Unexpected Dataset 3 shape: {adata.shape}")
if "counts" not in adata.layers:
    raise RuntimeError("Canonical counts layer is missing.")
if not adata.obs_names.is_unique:
    raise RuntimeError("Cell identifiers are not unique.")

print("scRareBench:", scrarebench.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print(adata)
print(info)


## 3. scDisInFact configuration

In [ ]:
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "n_hvg": 4000,
    "hvg_policy": "global_seurat_v3_raw_counts",
    "hvg_batch_key": None,
    "hvg_span": 0.3,

    "condition_column": "_scdisinfact_condition",
    "condition_value": "baseline",

    "Ks": [8, 4],
    "batch_size": 64,
    "interval": 10,
    "lr": 5e-4,
    "reg_mmd_comm": 1e-4,
    "reg_mmd_diff": 1e-4,
    "reg_gl": 1.0,
    "reg_class": 1.0,
    "reg_kl_comm": 1e-5,
    "reg_kl_diff": 1e-2,
    "enc_injection": False,
    "nepochs": 100,
    "recon_loss": "NB",
}

OUTPUT_DIR = "/content/scrarebench_scdisinfact_dataset3_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_scdisinfact_dataset3_method_runs"


## 4. User-owned scDisInFact runner

The runner performs only method-owned work: global HVG selection, scDisInFact
preprocessing/training, extraction of shared `mu_c`, and exact cell-order
restoration. Dataset preparation, scenarios, evaluation, scIB and reporting
remain package-owned.


In [ ]:
from pathlib import Path
import gc, json, random, shutil

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch

from scDisInFact import create_scdisinfact_dataset, scdisinfact
from scrarebench import MethodOutput, dataset_info

_BARCODE_COL = "_scrarebench_original_barcode"


def _count_guard(matrix, seed, max_values=100_000):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).reshape(-1)
    values = np.asarray(values)
    if values.size > max_values:
        rng = np.random.default_rng(int(seed))
        values = values[rng.choice(values.size, size=max_values, replace=False)]
    values = values.astype(float, copy=False)
    if values.size == 0:
        return
    if not np.isfinite(values).all():
        raise ValueError("Counts contain NaN/Inf.")
    if values.min() < 0:
        raise ValueError("Counts contain negatives.")
    if not np.allclose(values, np.rint(values), atol=1e-6, rtol=0):
        raise ValueError("Counts are not integer-like.")


def run_scdisinfact(method_adata, seed, config):
    method_info = dataset_info(method_adata)
    batch_key = method_info["batch_key"]
    count_layer = method_info["count_layer"]

    if method_info.get("dataset_key") != "wu_breast_cancer_atlas":
        raise RuntimeError("Runner is configured only for Dataset 3.")
    if method_info.get("scib_hvg_batch_mode") != "global":
        raise RuntimeError("Dataset 3 must use global HVG policy.")
    if not count_layer or count_layer not in method_adata.layers:
        raise KeyError(f"Missing canonical count layer {count_layer!r}.")
    if batch_key not in method_adata.obs.columns:
        raise KeyError(f"Missing batch column {batch_key!r}.")
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this Dataset 3 run.")

    original_order = method_adata.obs_names.astype(str).to_numpy(copy=True)
    if len(np.unique(original_order)) != len(original_order):
        raise RuntimeError("Cell identifiers must be unique.")

    _count_guard(method_adata.layers[count_layer], seed)

    # Dataset 3 fixed GLOBAL HVG policy: deliberately no batch_key.
    sc.pp.highly_variable_genes(
        method_adata,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(int(config["n_hvg"]), method_adata.n_vars),
        span=float(config["hvg_span"]),
        subset=False,
        check_values=True,
    )

    hvg_mask = method_adata.var["highly_variable"].fillna(False).to_numpy(bool)
    n_selected = int(hvg_mask.sum())
    expected_hvg = min(int(config["n_hvg"]), method_adata.n_vars)
    if n_selected != expected_hvg:
        raise RuntimeError(
            f"Expected {expected_hvg} global HVGs, selected {n_selected}."
        )

    counts_hvg = method_adata.layers[count_layer][:, hvg_mask]
    counts_hvg = (
        counts_hvg.tocsr().astype(np.float32)
        if sp.issparse(counts_hvg)
        else np.asarray(counts_hvg, dtype=np.float32)
    )

    cell_sums = np.asarray(counts_hvg.sum(axis=1)).reshape(-1)
    zero_cells = int(np.count_nonzero(cell_sums <= 0))
    if zero_cells:
        raise ValueError(
            f"{zero_cells} cells have zero counts across selected HVGs."
        )

    condition_col = str(config["condition_column"])

    # No ground-truth label is copied into method metadata.
    meta_cells = pd.DataFrame(index=method_adata.obs_names.copy())
    meta_cells[batch_key] = method_adata.obs[batch_key].astype(str).to_numpy()
    meta_cells[condition_col] = str(config["condition_value"])
    meta_cells[_BARCODE_COL] = original_order

    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))
    device = torch.device("cuda:0")

    seed_dir = Path(METHOD_WORK_DIR) / f"seed_{int(seed)}"
    if seed_dir.exists():
        shutil.rmtree(seed_dir)
    seed_dir.mkdir(parents=True, exist_ok=True)

    print(
        f"Seed {seed}: cells={method_adata.n_obs:,}; "
        f"HVGs={n_selected:,}; donors={meta_cells[batch_key].nunique()}; "
        f"GPU={torch.cuda.get_device_name(0)}"
    )

    data_dict = create_scdisinfact_dataset(
        counts_hvg,
        meta_cells,
        condition_key=[condition_col],
        batch_key=batch_key,
        batch_cond_key=None,
        log_trans=True,
    )

    model = scdisinfact(
        data_dict=data_dict,
        reg_mmd_comm=float(config["reg_mmd_comm"]),
        reg_mmd_diff=float(config["reg_mmd_diff"]),
        reg_gl=float(config["reg_gl"]),
        reg_class=float(config["reg_class"]),
        reg_kl_comm=float(config["reg_kl_comm"]),
        reg_kl_diff=float(config["reg_kl_diff"]),
        Ks=list(config["Ks"]),
        batch_size=int(config["batch_size"]),
        interval=int(config["interval"]),
        lr=float(config["lr"]),
        seed=int(seed),
        enc_injection=bool(config["enc_injection"]),
        device=device,
    )

    losses = model.train_model(
        nepochs=int(config["nepochs"]),
        recon_loss=str(config["recon_loss"]),
    )
    model.eval()

    latent_parts = []
    barcode_parts = []

    with torch.no_grad():
        for dataset_part, meta_part in zip(
            data_dict["datasets"],
            data_dict["meta_cells"],
        ):
            inf = model.inference(
                counts=dataset_part.counts_norm.to(model.device),
                batch_ids=dataset_part.batch_id[:, None].to(model.device),
                print_stat=False,
            )
            latent_parts.append(
                inf["mu_c"].detach().cpu().numpy().astype(np.float32, copy=False)
            )
            barcode_parts.append(
                meta_part[_BARCODE_COL].astype(str).to_numpy()
            )

    latent_native = np.concatenate(latent_parts, axis=0)
    native_barcodes = np.concatenate(barcode_parts, axis=0)

    expected_shape = (method_adata.n_obs, int(config["Ks"][0]))
    if latent_native.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected shared latent shape: {latent_native.shape}; "
            f"expected {expected_shape}."
        )
    if not np.isfinite(latent_native).all():
        raise ValueError("Shared latent contains NaN/Inf.")
    if len(np.unique(native_barcodes)) != len(native_barcodes):
        raise RuntimeError("Recovered barcodes are not unique.")
    if set(native_barcodes) != set(original_order):
        raise RuntimeError("scDisInFact changed the Dataset 3 cell set.")

    position = pd.Series(np.arange(len(native_barcodes)), index=native_barcodes)
    reorder = position.loc[original_order].to_numpy()
    latent = latent_native[reorder]

    if not np.array_equal(native_barcodes[reorder], original_order):
        raise RuntimeError("Failed to restore original cell order.")

    loss_names = [
        "total",
        "reconstruction",
        "kl_common",
        "kl_diff",
        "mmd_common",
        "mmd_diff",
        "classification",
        "group_lasso_diff",
    ]
    loss_path = seed_dir / "training_validation_losses.csv"
    pd.DataFrame(
        {name: values for name, values in zip(loss_names, losses)}
    ).to_csv(loss_path, index=False)

    config_path = seed_dir / "method_config.json"
    payload = dict(config)
    payload.update({
        "seed": int(seed),
        "selected_hvgs": n_selected,
        "upstream_commit": SCDISINFACT_COMMIT,
        "condition_policy": "constant baseline; evaluate shared mu_c",
        "evaluated_factor": "mu_c",
    })
    config_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    print(f"Seed {seed}: aligned shared latent={latent.shape}")

    del model, data_dict, latent_native, counts_hvg, position, reorder
    gc.collect()
    torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=original_order,
        representation_key="X_scdisinfact_shared",
        provenance_files={
            "scdisinfact_training_losses": loss_path,
            "scdisinfact_method_config": config_path,
        },
    )


## 5. Run the high-level benchmark

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scDisInFact",
    runner=run_scdisinfact,
    config=METHOD_CONFIG,
    dependencies=METHOD_RUNTIME_DEPENDENCIES + (SCDISINFACT_GITHUB,),
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        "write_pdf_report": False,
        "include_latent_in_bundle": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,
    copy_input_per_seed=True,
)


## 6. Inspect results

In [ ]:
from IPython.display import display

display(result.summary().round(5))
print("Method:", result.method)
print("Seeds:", result.seeds)
print("Multi-seed HTML:", result.report_path)
print("Archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(f"\nSeed {seed}")
    print("  representation:", run.representation_key)
    print("  report:", run.interactive_report_path)
    print("  bundle:", run.bundle_path)


## 7. Download final archive

In [ ]:
from pathlib import Path

archive = Path(result.archive_path)
if not archive.exists():
    raise FileNotFoundError(archive)

print("Archive:", archive)
print("Size (MB):", round(archive.stat().st_size / 1024 / 1024, 2))

try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass
